In [ ]:
#5band-freq
#3hb

In [1]:
#version2 - session_id
# MULTIMODAL PREPROCESSING: EEG + fNIRS 
# Set the paths based on your system before running the code
# you can adjust which subjects to process at the bottom of this cell

import os
import numpy as np
import scipy.io as sio
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, resample
from scipy.interpolate import griddata
from scipy.interpolate import CloughTocher2DInterpolator  # NEW: used by the vectorized spatial_mapping()
from scipy.spatial import Delaunay  # NEW: used by the vectorized spatial_mapping()
import mne
import mne_icalabel
import pywt
import scipy.stats as stats
import pandas as pd
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt  # NEW: needed for before/after preprocessing plots
import time
import datetime
import json  # NEW: for per-subject metadata.json export (adopted from reviewing a third-party implementation, see chat history)
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"  # NEW: matches user's actual dataset folder
OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"  # NEW: matches user's actual output folder

# window parameters
EEG_WINDOW_LEN = 3.0      
EEG_STEP = 1.0            # 66% overlap
EEG_EPOCH_TMIN = 0.0
EEG_EPOCH_TMAX = 40.0      # NEW: cover the full 40 s task period of a single n-back trial

FNIRS_WINDOW_LEN = 3.0    # CHANGED: 10.0 -> 3.0 (length of EACH of the 11 segments, not the whole span)
FNIRS_NUM_SEGMENTS = 11   # NEW: number of overlapping 3s segments per EEG window, per the paper's approach
FNIRS_STEP = 1.0          # 90% overlap

BASELINE_TMIN = -5.0  # NEW: seconds before trial onset (start of baseline window)
BASELINE_TMAX = -2.0  # NEW: seconds before trial onset (end of baseline window)
FNIRS_EPOCH_TMIN = 0.0
FNIRS_EPOCH_TMAX = 18.0   # bacause of the hemodynamic delay

SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0      

EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}      # NEW: EEG session/trial-onset codes -> class
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}          # NEW: fNIRS session/trial-onset codes -> class

# ============================================================================
# NEW (5-band EEG decomposition, per MSTFDN Ding et al. 2026, Information Fusion):
# instead of feeding broadband EEG into spatial_mapping, the cleaned EEG signal
# (post-ICA, post-reref) is decomposed into 5 standard frequency bands BEFORE
# spatial mapping. Each band is spatially mapped independently and the 5 resulting
# 16x16 grids are stacked on a new trailing axis.
# ============================================================================
EEG_FREQ_BANDS = {
    'delta': (0.5, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta':  (13.0, 30.0),
    'gamma': (30.0, 40.0),
}
EEG_BAND_ORDER = ['delta', 'theta', 'alpha', 'beta', 'gamma']  # NEW: fixed order used everywhere bands are stacked


def apply_band_decomposition(raw_clean):
    """
    ورودی: raw_clean (سیگنال تمیزشده‌ی broadband، بعد از ICA، bad-channel interpolation و
    average re-reference -- یعنی خروجی preprocess_eeg، raw_eeg_clean در process_subject).
    خروجی: دیکشنری {نام_باند: MNE Raw object فیلترشده}.
    نکته: این تابع فقط تعریف می‌شود اینجا؛ فراخوانی واقعی‌اش در process_subject روی
    raw_eeg_clean انجام می‌شود (بعد از drop شدن VEOG/HEOG/TP9)، نه داخل preprocess_eeg،
    تا channel order با eeg_mnt (که بعد از preprocess_eeg ساخته می‌شود) هماهنگ بماند.
    """
    band_signals = {}
    for band_name, (l_freq, h_freq) in EEG_FREQ_BANDS.items():
        raw_band = raw_clean.copy().filter(
            l_freq=l_freq, h_freq=h_freq, method='fir', verbose=False
        )
        band_signals[band_name] = raw_band
    return band_signals


# LOAD EEG DATA
def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    
    print(f"Loading EEG from: {cnt_path}")
    
    # Load signal
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]
    cnt_key = real_cnt_keys[0]
    cnt_struct = mat_cnt[cnt_key]
    
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    
    if sfreq != SFREQ_EEG:
        print(f"Resampling from {sfreq} to {SFREQ_EEG} Hz")
        from scipy.signal import resample
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')
    
    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    real_mrk_keys = [k for k in mat_mrk.keys() if not k.startswith('__')]
    mrk_key = real_mrk_keys[0]
    mrk_struct = mat_mrk[mrk_key]
    
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    elif hasattr(mrk_struct, 'y'):
        y = mrk_struct.y
        if y.ndim > 1:
            if y.shape[0] == 1 or y.shape[1] == 1:
                event_codes = y.flatten()
            else:
                event_codes = np.argmax(y, axis=0) + 1
        else:
            event_codes = y.flatten()
    else:
        event_codes = event_positions.copy()
    
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes
    
    print(f"EEG loaded: {raw.n_times} samples, {sfreq} Hz, {len(events)} markers")
    print(f"unique markers: {np.unique(event_codes)}")

    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')  # NEW
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_eeg_mnt_keys = [k for k in mat_eeg_mnt.keys() if not k.startswith('__')]  # NEW
    eeg_mnt_struct = mat_eeg_mnt[real_eeg_mnt_keys[0]]  # NEW: has .pos_3d (3,30) and .clab (30,)
    print(f"EEG mnt_nback.mat loaded: pos_3d shape {np.asarray(eeg_mnt_struct.pos_3d).shape}")  # NEW

    return raw, events, sfreq, eeg_mnt_struct  # CHANGED: also return the real electrode-position struct


# LOAD AND PREPROCESS FNIRS DATA
def load_and_preprocess_fnirs(subject_id):
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")  # NEW: flat folder structure
    
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')  # NEW
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")
    
    print(f"Loading raw fNIRS from: {cnt_path}")
    
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]     # NEW
    cnt = mat_cnt[real_cnt_keys[0]]                                          # NEW

    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith('__')]     # NEW
    mnt = mat_mnt[real_mnt_keys[0]]                                          # NEW

    fnirs_events_time = None  # NEW
    fnirs_events_code = None  # NEW
    if os.path.exists(mrk_path):  # NEW
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)  # NEW
        real_fmrk_keys = [k for k in mat_fmrk.keys() if not k.startswith('__')]    # NEW
        fmrk = mat_fmrk[real_fmrk_keys[0]]                                         # NEW
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):  # NEW
            fnirs_events_time = np.asarray(fmrk.time).flatten()                     # NEW
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()               # NEW
    
    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0  # NEW
    print(f"raw fNIRS sampling rate: {sfreq_fnirs_raw} Hz")
    
    oxy_raw = np.asarray(cnt.oxy.x, dtype=np.float64)     # NEW
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)  # NEW
    
    print(f"Raw fNIRS: oxy {oxy_raw.shape}, deoxy {deoxy_raw.shape}")
    
    print("Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...")  # NEW
    hbo_raw, hbr_raw = oxy_raw, deoxy_raw  # NEW
    
    print("Skipping wavelet/MARA/lowpass filtering (MSTFDN-style: relying on the model's own large-kernel convolutions for implicit smoothing)...")  # NEW
    hbo_filtered, hbr_filtered = hbo_raw, hbr_raw  # NEW
    
    print(f"Resampling from {sfreq_fnirs_raw} Hz to {SFREQ_FNIRS} Hz...")
    from scipy.signal import resample
    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)
    
    print("Spatial mapping to 16x16...")
    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)
    
    print(f"fNIRS preprocessed: {fnirs_data.shape}")

    try:  # NEW
        qc_channel = 0  # NEW
        t_raw = np.arange(oxy_raw.shape[0]) / sfreq_fnirs_raw  # NEW
        t_proc = np.arange(hbo_resampled.shape[0]) / SFREQ_FNIRS  # NEW
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)  # NEW
        axes[0].plot(t_raw, oxy_raw[:, qc_channel], color='tab:red', linewidth=0.6)  # NEW
        axes[0].set_title(f"{subject_id} - fNIRS RAW HbO (channel {qc_channel}, before preprocessing)")  # NEW
        axes[0].set_xlabel("Time (s)")  # NEW
        axes[0].set_ylabel("HbO (raw units)")  # NEW
        axes[1].plot(t_proc, hbo_resampled[:, qc_channel], color='tab:blue', linewidth=0.8)  # NEW
        axes[1].set_title(f"{subject_id} - fNIRS PREPROCESSED HbO (channel {qc_channel}, after wavelet+MARA+lowpass+resample)")  # NEW
        axes[1].set_xlabel("Time (s)")  # NEW
        axes[1].set_ylabel("HbO (mmol/L)")  # NEW
        plt.tight_layout()  # NEW
        qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
        os.makedirs(qc_dir, exist_ok=True)  # NEW
        fig.savefig(os.path.join(qc_dir, f"{subject_id}_fnirs_before_after.png"), dpi=150)  # NEW
        plt.close(fig)  # NEW
        print(f"Saved fNIRS before/after QC plot for {subject_id}")  # NEW
    except Exception as e:  # NEW
        print(f"Could not save fNIRS QC plot for {subject_id}: {e}")  # NEW
    
    return fnirs_data, fnirs_events_time, fnirs_events_code  # CHANGED

# fNIRS HELPER FUNCTIONS
def calculate_hbo_hbr(oxy, deoxy):
    lambda1, lambda2 = 760, 850
    L = 3.0
    
    def get_dpf(lmbda, age=25):
        alpha, beta, gamma = 223.3, 0.05624, 0.8493
        delta, eps, zeta = -5.723e-7, 0.001245, -0.9025
        return alpha + beta * (age ** gamma) + delta * (lmbda ** 3) + eps * (lmbda ** 2) + zeta * lmbda
    
    dpf1 = get_dpf(lambda1)
    dpf2 = get_dpf(lambda2)
    
    e1_hbo, e1_hbr = 0.586, 1.54852
    e2_hbo, e2_hbr = 1.058, 0.69132
    
    denom = L * (e1_hbr * e2_hbo - e2_hbr * e1_hbo)
    hbo = (e1_hbr * (deoxy / dpf2) - e2_hbr * (oxy / dpf1)) / denom
    hbr = (e1_hbo * (oxy / dpf1) - e2_hbo * (deoxy / dpf2)) / denom
    
    return hbo, hbr

def wavelet_denoising(data, alpha=0.1, wavelet='db5'):
    cleaned = np.zeros_like(data)
    n_samples, n_channels = data.shape
    
    for ch in range(n_channels):
        signal = data[:, ch]
        level = min(int(np.floor(np.log2(n_samples))), 6)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        new_coeffs = [coeffs[0]]
        for i in range(1, len(coeffs)):
            d_coeffs = coeffs[i].copy()
            sigma = np.median(np.abs(d_coeffs - np.median(d_coeffs))) / 0.6745
            if sigma > 0:
                threshold = sigma * stats.norm.ppf(1 - alpha / 2)
                d_coeffs[np.abs(d_coeffs) > threshold] = 0
            new_coeffs.append(d_coeffs)
        cleaned[:, ch] = pywt.waverec(new_coeffs, wavelet)[:n_samples]
    
    return cleaned

def apply_mara(data, fs=10.0, window_size=41, threshold_factor=1.8):
    n_samples, n_channels = data.shape
    corrected = np.zeros_like(data)
    edge_len = int(0.5 * fs)
    
    for ch in range(n_channels):
        signal = data[:, ch].copy()
        msd = pd.Series(signal).rolling(window=window_size, center=True, min_periods=1).std().to_numpy()
        threshold = np.median(msd) + (threshold_factor * np.std(msd))
        ma_mask = (msd > threshold).astype(int)
        
        diff_mask = np.diff(ma_mask, prepend=0, append=0)
        starts = np.where(diff_mask == 1)[0]
        ends = np.where(diff_mask == -1)[0]
        indices = sorted(list(set([0, n_samples] + list(starts) + list(ends))))
        segments = [signal[indices[i]:indices[i+1]].copy() for i in range(len(indices) - 1)]
        
        if not segments:
            corrected[:, ch] = signal
            continue
        
        final_parts = [segments[0]]
        for i in range(1, len(segments)):
            prev_seg = final_parts[-1]
            curr_seg = segments[i]
            val_prev = np.median(prev_seg[-edge_len:]) if len(prev_seg) >= edge_len else prev_seg[-1]
            val_curr = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
            shift = val_prev - val_curr
            
            if ma_mask[indices[i]] == 1:
                x = np.arange(len(curr_seg))
                s_val = max(np.var(curr_seg) * len(curr_seg), 1e-6)
                try:
                    spline = UnivariateSpline(x, curr_seg, s=s_val)
                    curr_seg = curr_seg - spline(x)
                    val_curr_new = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
                    curr_seg += (val_prev - val_curr_new)
                except:
                    curr_seg += shift
            else:
                if abs(shift) > 2 * np.std(msd):
                    curr_seg += shift
            final_parts.append(curr_seg)
        
        full_signal = np.concatenate(final_parts)[:n_samples]
        t_idx = np.arange(n_samples)
        original_trend = np.polyval(np.polyfit(t_idx, data[:, ch], 1), t_idx)
        corrected_trend = np.polyval(np.polyfit(t_idx, full_signal, 1), t_idx)
        full_signal += (original_trend - corrected_trend)
        corrected[:, ch] = full_signal
    
    return corrected

def lowpass_filter(data, fs=10.0, cutoff=0.15):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(3, normal_cutoff, btype='low')
    filtered = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered[:, ch] = filtfilt(b, a, data[:, ch])
    return filtered

def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)  # NEW: shape (3, n_channels)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]  # NEW
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)  # NEW
    r_safe = np.where(r > 0, r, 1.0)  # NEW
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))  # NEW
    phi = np.arctan2(y3d, x3d)  # NEW
    x_coords = theta * np.cos(phi)  # NEW
    y_coords = theta * np.sin(phi)  # NEW
    
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, 
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    
    tri = Delaunay(points)  # NEW: triangulation depends only on `points`, computed ONCE instead of n_samples times
    values = data.T  # NEW: (n_samples, n_channels) -> (n_channels, n_samples)
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)  # NEW
    result = interp((grid_x, grid_y))  # NEW: shape (16, 16, n_samples)
    mapped = np.transpose(result, (2, 0, 1)).astype(np.float32)  # NEW: (16, 16, n_samples) -> (n_samples, 16, 16)
    
    return mapped


# NEW (5-band spatial mapping, optimized): instead of calling spatial_mapping() 5 separate
# times (one per band, each re-computing the Delaunay triangulation + CloughTocher setup from
# scratch), this batches all 5 bands' worth of time samples into ONE interpolator call, since
# CloughTocher2DInterpolator accepts `values` with arbitrary trailing dimensions as long as
# values.shape[0] == n_points. Triangulation and the (expensive) gradient-estimation/patch-setup
# step of CloughTocher are therefore computed ONCE for all 5 bands combined, instead of 5x.
# Verified to be mathematically identical to calling spatial_mapping() once per band and stacking
# the results (same triangulation, same interpolant, only how it's batched changes).
def spatial_mapping_multiband(band_data_list, mnt):
    """
    ورودی:
        band_data_list: لیستی از ۵ آرایه، هرکدوم به شکل (n_times, n_channels) -- یک آرایه به ازای
                         هر باند، به همون ترتیب EEG_BAND_ORDER.
        mnt: همون آبجکتی که spatial_mapping() هم استفاده می‌کنه (دارای .pos_3d).
    خروجی:
        آرایه‌ی (n_times, 16, 16, n_bands) -- دقیقاً معادل با اینکه spatial_mapping() رو
        جداگانه برای هر باند صدا بزنیم و با np.stack(..., axis=-1) ترکیب‌شون کنیم، ولی با
        triangulation و patch-setup مشترک (یک‌بار محاسبه به‌جای n_bands بار).
    """
    n_bands = len(band_data_list)
    n_samples, n_channels = band_data_list[0].shape

    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)
    r_safe = np.where(r > 0, r, 1.0)
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))
    phi = np.arctan2(y3d, x3d)
    x_coords = theta * np.cos(phi)
    y_coords = theta * np.sin(phi)

    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j,
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))

    # (n_channels, n_times, n_bands) -- values.shape[0] must equal n_points (n_channels)
    stacked = np.stack(band_data_list, axis=-1)          # (n_times, n_channels, n_bands)
    values = np.transpose(stacked, (1, 0, 2))              # (n_channels, n_times, n_bands)

    tri = Delaunay(points)                                  # computed ONCE for all bands
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)  # gradient/patch setup ONCE for all bands
    result = interp((grid_x, grid_y))                       # (16, 16, n_times, n_bands)
    mapped = np.transpose(result, (2, 0, 1, 3)).astype(np.float32)  # (n_times, 16, 16, n_bands)

    return mapped


class _EEGMontageAdapter:  # NEW
    """Minimal stand-in for the fNIRS .mat mnt struct, exposing only the .pos_3d attribute that
    spatial_mapping() reads."""
    def __init__(self, pos_3d):  # NEW
        self.pos_3d = pos_3d  # NEW: shape (3, n_channels), matching mnt.pos_3d's shape for fNIRS

def get_eeg_mnt(raw, eeg_mnt_struct):  # CHANGED
    """
    Builds an _EEGMontageAdapter (pos_3d shape (3, n_channels)) from the REAL electrode 3D
    positions in the dataset's own EEG mnt_nback.mat (eeg_mnt_struct.pos_3d / .clab), selecting
    and ordering them to match raw.ch_names exactly. Must be called AFTER any channel drops
    (VEOG/HEOG/TP9 in preprocess_eeg) so the returned position order matches the channel order of
    the data that will be passed to spatial_mapping()/spatial_mapping_multiband().
    """
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]  # NEW
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)  # NEW: shape (3, 30)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}  # NEW
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}  # NEW

    pos_list = []  # NEW
    missing = []  # NEW
    for ch in raw.ch_names:  # NEW
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))  # NEW
        if idx is None:  # NEW
            missing.append(ch)  # NEW
            pos_list.append([0.0, 0.0, 0.0])  # NEW
        else:  # NEW
            pos_list.append(full_pos_3d[:, idx])  # NEW

    if missing:  # NEW
        print(f"  WARNING: {len(missing)} EEG channel(s) not found in mnt_nback.mat's clab "  # NEW
              f"(spatial_mapping will place them at the grid center): {missing}")  # NEW

    pos_3d = np.array(pos_list, dtype=np.float64).T  # NEW: shape (3, n_channels)
    return _EEGMontageAdapter(pos_3d)  # NEW

# PREPROCESS EEG
def preprocess_eeg(raw, events):
    print("Preprocessing EEG...")

    raw_before_qc = raw.copy()  # NEW
    
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks='eeg', verbose=False)
    # FIX (5-band consistency, per chat history): l_freq lowered from 1.0 -> 0.5 Hz so the
    # broadband signal that band-decomposition later slices from actually CONTAINS the 0.5-1 Hz
    # content that the delta band (0.5-4 Hz, MSTFDN definition) needs. With l_freq=1.0, the delta
    # band filter would silently produce a 1-4 Hz band instead of the intended 0.5-4 Hz band,
    # since content below 1 Hz was already removed here. This also matches MSTFDN's own broadband
    # preprocessing range. Gamma (30-40 Hz) is unaffected since h_freq=40.0 is unchanged. ICA runs
    # AFTER this filter, so lowering l_freq may let more slow baseline-wander survive into ICA;
    # verify on 2-3 subjects that the number of ICA components removed doesn't change drastically
    # before running the full 26-subject batch (see verification checklist).
    raw_filtered.filter(l_freq=0.5, h_freq=40.0, method='fir', verbose=False)  # FIX: 1.0 -> 0.5
    
    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method='fir', verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ['VEOG', 'HEOG', 'TP9']:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    
    if bad_channels:
        print(f"Bad channels: {bad_channels}")
        raw_filtered.info['bads'] = bad_channels
    
    interpolated_channels = list(raw_filtered.info['bads'])  # NEW
    
    if raw_filtered.info['bads']:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    
    raw_filtered.set_eeg_reference('average', projection=False, verbose=False)
    
    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude='bads')

    print(f"ICA: using variance-based component selection (99% explained variance), "  # NEW
          f"picks={len(picks_for_ica)}, interpolated={len(interpolated_channels)}")     # NEW
    ica = mne.preprocessing.ICA(n_components=0.99, method='picard',                     # NEW
                                fit_params=dict(ortho=False, extended=True),            # NEW
                                random_state=42, verbose=False)                          # NEW
    ica.fit(raw_filtered, picks=picks_for_ica)

    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method='iclabel')

    ARTIFACT_CLASSES_IDX = {  # NEW
        'brain': 0, 'muscle artifact': 1, 'eye blink': 2,
        'heart beat': 3, 'line noise': 4, 'channel noise': 5, 'other': 6
    }
    ARTIFACT_TARGETS = ['eye blink', 'eye movement', 'muscle artifact', 'line noise']  # NEW
    PROB_THRESHOLD = 0.30  # NEW

    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels['labels'])
                           if label in ARTIFACT_TARGETS]  # CHANGED

    artifact_idx_prob = []  # NEW
    proba_matrix = ica.labels_scores_  # NEW
    for comp_idx in range(proba_matrix.shape[0]):  # NEW
        for cls_name in ARTIFACT_TARGETS:  # NEW
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)  # NEW
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:  # NEW
                artifact_idx_prob.append(comp_idx)  # NEW
                break  # NEW

    eog_indices_veog, eog_scores_veog = [], None  # NEW
    eog_indices_heog, eog_scores_heog = [], None  # NEW
    try:  # NEW
        if 'VEOG' in raw.ch_names:  # NEW
            eog_indices_veog, eog_scores_veog = ica.find_bads_eog(raw, ch_name='VEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(VEOG) flagged components: {eog_indices_veog}")  # NEW
        if 'HEOG' in raw.ch_names:  # NEW
            eog_indices_heog, eog_scores_heog = ica.find_bads_eog(raw, ch_name='HEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(HEOG) flagged components: {eog_indices_heog}")  # NEW
    except Exception as e:  # NEW
        print(f"  find_bads_eog failed: {e}")  # NEW

    BRAIN_PROB_CEILING = 0.5  # NEW
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX['brain']  # NEW
    
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))  # NEW
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}  # NEW
    if eog_only_rescued:  # NEW
        print(f"  Rescued from EOG-only rejection (ICLabel brain-prob >= {BRAIN_PROB_CEILING}): {sorted(eog_only_rescued)}")  # NEW
    
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )  # CHANGED

    all_class_names = ['brain', 'muscle artifact', 'eye blink', 'heart beat', 'line noise', 'channel noise', 'other']  # NEW
    for comp_idx in artifact_idx:  # NEW
        probs_str = ", ".join([f"{all_class_names[c]}={proba_matrix[comp_idx, c]:.2f}"  # NEW
                               for c in range(7)])  # NEW
        reason = []  # NEW
        if comp_idx in artifact_idx_argmax:  # NEW
            reason.append(f"argmax='{ic_labels['labels'][comp_idx]}'")  # NEW
        if comp_idx in artifact_idx_prob:  # NEW
            reason.append(f"prob>={PROB_THRESHOLD}")  # NEW
        if comp_idx in eog_indices_veog:  # NEW
            reason.append("VEOG-correlation")  # NEW
        if comp_idx in eog_indices_heog:  # NEW
            reason.append("HEOG-correlation")  # NEW
        print(f"  IC{comp_idx} flagged [{', '.join(reason)}]: {probs_str}")  # NEW

    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components")
    else:
        print("Removed 0 ICA components (no artifact ICs detected)")  # NEW

    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ['VEOG', 'HEOG', 'TP9']]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    try:  # NEW
        def _plot_all_channels_stacked(raw_obj, title, color, tmin=None, tmax=None):  # CHANGED
            sfreq_plot = raw_obj.info['sfreq']               # NEW
            if tmin is not None or tmax is not None:         # NEW
                start_sample = int((tmin or 0.0) * sfreq_plot)             # NEW
                stop_sample = int((tmax * sfreq_plot)) if tmax is not None else None  # NEW
                data = raw_obj.get_data(start=start_sample, stop=stop_sample)         # NEW
                t = (np.arange(data.shape[1]) + start_sample) / sfreq_plot            # NEW
            else:                                            # NEW
                data = raw_obj.get_data()                      # NEW
                t = np.arange(data.shape[1]) / sfreq_plot        # NEW
            ch_names_plot = raw_obj.ch_names                # NEW

            spread = np.median(np.abs(data - np.median(data, axis=1, keepdims=True))) * 30 + 1e-12  # NEW
            n_ch_plot = data.shape[0]                        # NEW
            fig_h = max(6, 0.5 * n_ch_plot)                 # NEW
            fig, ax = plt.subplots(figsize=(14, fig_h))      # NEW
            for i in range(n_ch_plot):                       # NEW
                offset = (n_ch_plot - 1 - i) * spread         # NEW
                ax.plot(t, data[i] + offset, color=color, linewidth=0.4)  # NEW
            ax.set_yticks([(n_ch_plot - 1 - i) * spread for i in range(n_ch_plot)])  # NEW
            ax.set_yticklabels(ch_names_plot, fontsize=6)    # NEW
            ax.set_xlabel("Time (s)")                        # NEW
            ax.set_ylabel("Channel")                         # NEW
            ax.set_title(title)                              # NEW
            plt.tight_layout()                                # NEW
            return fig                                        # NEW

        fig_before = _plot_all_channels_stacked(  # NEW
            raw_before_qc, "EEG RAW - All Channels (before preprocessing)", 'tab:red'  # NEW
        )  # NEW
        fig_after = _plot_all_channels_stacked(  # NEW
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA)", 'tab:blue'  # NEW
        )  # NEW
        fig_before_zoom = _plot_all_channels_stacked(  # NEW
            raw_before_qc, "EEG RAW - All Channels (before preprocessing, 10-30s)", 'tab:red', tmin=10.0, tmax=30.0  # NEW
        )  # NEW
        fig_after_zoom = _plot_all_channels_stacked(  # NEW
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA, 10-30s)", 'tab:blue', tmin=10.0, tmax=30.0  # NEW
        )  # NEW

        tmp_qc_dir = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc")  # NEW
        os.makedirs(tmp_qc_dir, exist_ok=True)  # NEW
        fig_before.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels.png"), dpi=150)  # NEW
        fig_after.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels.png"), dpi=150)  # NEW
        fig_before_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels_zoom10_30.png"), dpi=150)  # NEW
        fig_after_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels_zoom10_30.png"), dpi=150)  # NEW
        plt.close(fig_before)  # NEW
        plt.close(fig_after)  # NEW
        plt.close(fig_before_zoom)  # NEW
        plt.close(fig_after_zoom)  # NEW
    except Exception as e:  # NEW
        print(f"Could not save EEG QC plot: {e}")  # NEW
    
    return raw_filtered


def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,  # CHANGED
                                  fnirs_events_time, fnirs_events_code):  # CHANGED

    print("Extracting synchronized asymmetric sliding windows...")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    
    label_map = EEG_TRIAL_LABEL_MAP  # NEW
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())  # NEW
    target_events = events[np.isin(events[:, 2], valid_codes)]  # NEW
    
    print(f"found {len(target_events)} trial markers")
    if len(target_events) == 0:
        print(f"No trial markers found. Available markers: {np.unique(events[:, 2])}")
        return np.array([]), np.array([]), np.array([])

    fnirs_sync_ok = False  # NEW
    fnirs_trial_times = None  # NEW
    if fnirs_events_time is not None and fnirs_events_code is not None:  # NEW
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())  # NEW
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)  # NEW
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)  # NEW
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]  # NEW
        fnirs_order = np.argsort(fnirs_trial_times_ms)  # NEW
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0  # NEW
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])  # NEW

        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])  # NEW
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))  # NEW
        if (len(fnirs_trial_times) == len(target_events)  # NEW
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):  # NEW
            fnirs_sync_ok = True  # NEW
            print(f"fNIRS-own-marker sync check: OK ({len(fnirs_trial_times)} trials, class order matches EEG)")  # NEW
        else:
            print(f"  WARNING: fNIRS own markers do not match EEG 1:1 "  # NEW
                  f"(fNIRS n={len(fnirs_trial_times)}, EEG n={len(target_events)}) -- "  # NEW
                  f"falling back to EEG-clock-derived fNIRS timing for this subject.")  # NEW
    else:
        print("  WARNING: fNIRS event markers unavailable -- falling back to EEG-clock-derived fNIRS timing.")  # NEW
    
    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP
    
    eeg_windows = []
    fnirs_windows = []
    labels = []
    session_ids = []   # CROSS-SESSION
    trial_ids = []     # NEW: needed by training script's trial-level split (VPxxx_trial_ids.npy)
    
    for trial_idx, ev in enumerate(target_events):  # CHANGED
        event_sample_eeg = ev[0]
        raw_code = ev[2]
        
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        TRIALS_PER_SESSION = 9
        session_id = trial_idx // TRIALS_PER_SESSION + 1
        
        eeg_event_time = event_sample_eeg / sfreq_eeg  # CHANGED
        if fnirs_sync_ok:  # NEW
            fnirs_event_time = fnirs_trial_times[trial_idx]  # NEW
        else:  # NEW
            fnirs_event_time = eeg_event_time  # NEW

        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))  # NEW
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))    # NEW
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:  # NEW
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)  # NEW
        else:  # NEW
            fnirs_baseline = 0.0  # NEW

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            
            if w_end_eeg > eeg_raw.n_times:  # CHANGED
                continue

            fnirs_segments = []  # NEW
            valid_segments = True  # NEW
            for seg_idx in range(FNIRS_NUM_SEGMENTS):  # NEW
                seg_offset = offset_sec + seg_idx * FNIRS_STEP  # NEW
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))  # NEW
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs  # NEW

                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:  # NEW
                    valid_segments = False  # NEW
                    break  # NEW

                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)  # CHANGED

            if not valid_segments:  # NEW
                continue  # NEW
            
            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)  # CHANGED: QC-only now
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:  # CHANGED
                continue

            # CHANGED (5-band EEG): eeg_spatial_data is now (n_times, 16, 16, 5) -- one grid per
            # band, stacked on the trailing axis (see spatial_mapping_multiband() / EEG_BAND_ORDER).
            # Slicing along the time axis (axis 0) is unaffected by the extra trailing band axis.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # CHANGED: (600, 16, 16, 5)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 3, 0)          # CHANGED: (16, 16, 5, 600)

            fnirs_slice = np.stack(fnirs_segments, axis=0)      # NEW: (11, 30, 16, 16, 2)
            fnirs_slice = fnirs_slice.transpose(0, 2, 3, 1, 4)  # NEW: (11, 16, 16, 30, 2)
            
            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
            session_ids.append(session_id)   # CROSS-SESSION
            trial_ids.append(trial_idx)       # NEW
    
    eeg_windows = np.array(eeg_windows)
    fnirs_windows = np.array(fnirs_windows)
    labels = np.array(labels)
    session_ids = np.array(session_ids, dtype=np.int32)   # CROSS-SESSION
    trial_ids = np.array(trial_ids, dtype=np.int32)        # NEW
    
    print(f"Extracted {len(labels)} synchronized windows")
    if len(labels) > 0:
        print(f"Label distribution: {np.bincount(labels)}")
        print(f"EEG windows shape: {eeg_windows.shape} (expect (N,16,16,5,600))")  # NEW: sanity print
    
    return eeg_windows, fnirs_windows, labels, session_ids, trial_ids   # CHANGED: trial_ids added



LOG_FILE = os.path.join(OUTPUT_PATH, "processing_log.txt")  # NEW

def log_message(msg):  # NEW
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")



# MAIN PROCESSING FUNCTION
def process_subject(subject_id):
    t0 = time.time()  # NEW
    log_message(f"START processing {subject_id}")  # NEW
    print(f"Processing {subject_id}")
    
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")  # NEW
    
    if not os.path.exists(eeg_folder):
        log_message(f"SKIPPED {subject_id}: EEG folder not found")  # NEW
        print(f"EEG folder not found: {eeg_folder}")
        return
    
    try:
        print("\n Loading EEG...")
        raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)  # CHANGED
        
        print("\n Loading and preprocessing fNIRS...")
        fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(subject_id)  # CHANGED
        
        print("\n Normalizing fNIRS (subject-level, before windowing)...")  # NEW
        fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)  # NEW
        
        print("\n Preprocessing EEG...")
        raw_eeg_clean = preprocess_eeg(raw_eeg, events)

        eeg_full_data = raw_eeg_clean.get_data()  # NEW: kept for informational print only (channel-space scale)
        eeg_subject_mean = eeg_full_data.mean(axis=-1, keepdims=True)  # NEW
        eeg_subject_std = eeg_full_data.std(axis=-1, keepdims=True) + 1e-8  # NEW
        print(f"\n EEG per-subject (channel-space) stats: "  # NEW
              f"mean range [{eeg_subject_mean.min():.2e}, {eeg_subject_mean.max():.2e}], "  # NEW
              f"std range [{eeg_subject_std.min():.2e}, {eeg_subject_std.max():.2e}]")  # NEW

        # CHANGED (5-band EEG decomposition, per MSTFDN Ding et al. 2026): instead of spatially
        # mapping the single broadband signal, raw_eeg_clean is first decomposed into 5 standard
        # frequency bands (delta/theta/alpha/beta/gamma), each spatially mapped to a 16x16 grid,
        # and the 5 grids are stacked on a trailing axis -> (n_times, 16, 16, 5). Uses the batched
        # spatial_mapping_multiband() so the (expensive) Delaunay triangulation + CloughTocher
        # patch-setup is computed ONCE for all 5 bands instead of 5 separate times.
        print("\n Decomposing EEG into 5 frequency bands (delta/theta/alpha/beta/gamma)...")  # NEW
        band_signals = apply_band_decomposition(raw_eeg_clean)  # NEW: dict {band_name: Raw}

        eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)  # CHANGED: uses REAL mnt_nback.mat positions, matched by clab name

        print("\n Spatial mapping EEG (5 bands) to 16x16...")  # CHANGED
        band_data_list = []  # NEW
        for band_name in EEG_BAND_ORDER:  # NEW
            raw_band = band_signals[band_name]  # NEW
            band_data = raw_band.get_data()  # NEW: (n_channels, n_times)
            # per-channel z-score of this band's continuous signal, BEFORE spatial mapping,
            # mirroring the normalization already applied to broadband EEG in the previous pipeline
            band_data_normalized = (band_data - band_data.mean(axis=-1, keepdims=True)) / (  # NEW
                band_data.std(axis=-1, keepdims=True) + 1e-8)  # NEW
            band_data_list.append(band_data_normalized.T)  # NEW: (n_times, n_channels), matches spatial_mapping's expected input order

        eeg_spatial_full = spatial_mapping_multiband(band_data_list, eeg_mnt)  # NEW: (n_times, 16, 16, 5)
        print(f"EEG spatially mapped (5 bands): {eeg_spatial_full.shape}")  # NEW

        # per-band, per-grid-cell normalization of the spatially-mapped signal, applied to the
        # FULL continuous grid before windowing (mirrors fNIRS normalization above). axis=0 is
        # time, so mean/std are computed per (row, col, band) cell across the whole recording.
        eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (  # CHANGED
            eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)  # CHANGED

        try:  # NEW
            tmp_qc_before = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels.png")  # NEW
            tmp_qc_after = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels.png")    # NEW
            tmp_qc_before_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels_zoom10_30.png")  # NEW
            tmp_qc_after_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels_zoom10_30.png")    # NEW
            if os.path.exists(tmp_qc_before) and os.path.exists(tmp_qc_after):  # NEW
                final_qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
                os.makedirs(final_qc_dir, exist_ok=True)  # NEW
                final_qc_before = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels.png")  # NEW
                final_qc_after = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels.png")    # NEW
                os.replace(tmp_qc_before, final_qc_before)  # NEW
                os.replace(tmp_qc_after, final_qc_after)    # NEW
                if os.path.exists(tmp_qc_before_zoom) and os.path.exists(tmp_qc_after_zoom):  # NEW
                    final_qc_before_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels_zoom10_30.png")  # NEW
                    final_qc_after_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels_zoom10_30.png")    # NEW
                    os.replace(tmp_qc_before_zoom, final_qc_before_zoom)  # NEW
                    os.replace(tmp_qc_after_zoom, final_qc_after_zoom)    # NEW
                print(f"Saved EEG before/after (all-channels) QC plots for {subject_id}")  # NEW
        except Exception as e:  # NEW
            print(f"Could not finalize EEG QC plot for {subject_id}: {e}")  # NEW
        
        print("\n Extracting synchronized asymmetric windows...")
        eeg_windows, fnirs_windows, labels, session_ids, trial_ids = extract_synchronized_windows(   # CHANGED: trial_ids added
            raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg,
            fnirs_events_time, fnirs_events_code
        )
        
        if len(labels) == 0:
            log_message(f"WARNING {subject_id}: no windows extracted")  # NEW
            print(f"No windows extracted for {subject_id}")
            return
        
        print("\n Normalizing...")
        # NOTE: EEG normalization already happened above (per-band, per-grid-cell, before
        # windowing) -- eeg_windows already carries normalized values, no further step needed here.
        
        print("\n Saving...")
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
        os.makedirs(sub_out_dir, exist_ok=True)
        
        np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)    # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)  # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)               # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_session_ids.npy"), session_ids)   # CROSS-SESSION
        np.save(os.path.join(sub_out_dir, f"{subject_id}_trial_ids.npy"), trial_ids)         # NEW

        metadata = {  # NEW
            'subject_id': subject_id,  # NEW
            'eeg_shape': list(eeg_windows.shape),  # NEW: expect (N,16,16,5,600)
            'fnirs_shape': list(fnirs_windows.shape),  # NEW
            'labels_distribution': np.bincount(labels).tolist(),  # NEW
            'eeg_window_len': EEG_WINDOW_LEN,  # NEW
            'eeg_step': EEG_STEP,  # NEW
            'eeg_epoch_tmax': EEG_EPOCH_TMAX,  # NEW
            'eeg_freq_bands': EEG_FREQ_BANDS,  # NEW
            'eeg_band_order': EEG_BAND_ORDER,  # NEW
            'eeg_broadband_lfreq': 0.5,  # NEW: documents the 1.0 -> 0.5 fix
            'fnirs_num_segments': FNIRS_NUM_SEGMENTS,  # NEW
            'fnirs_window_len': FNIRS_WINDOW_LEN,  # NEW
            'fnirs_step': FNIRS_STEP,  # NEW
            'sfreq_eeg': SFREQ_EEG,  # NEW
            'sfreq_fnirs': SFREQ_FNIRS,  # NEW
            'beer_lambert_applied': False,  # NEW
            'fnirs_sync_fix_applied': True,  # NEW
            'normalization': 'per-subject z-score (EEG grid per-band + fNIRS grid, before windowing)',  # CHANGED
        }
        with open(os.path.join(sub_out_dir, f"{subject_id}_metadata.json"), 'w') as f:  # NEW
            json.dump(metadata, f, indent=2)
        
        print(f"\n {subject_id} completed!")
        print(f"EEG windows: {eeg_windows.shape}")
        print(f"fNIRS windows: {fnirs_windows.shape}")
        print(f"Labels: {np.bincount(labels)}")
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"DONE {subject_id} in {elapsed:.1f} min | EEG: {eeg_windows.shape} | fNIRS: {fnirs_windows.shape} | Labels: {np.bincount(labels)}")  # NEW
        
    except Exception as e:
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"ERROR processing {subject_id} after {elapsed:.1f} min: {str(e)}")  # NEW
        print(f"Error processing {subject_id}: {str(e)}")
        import traceback
        traceback.print_exc()

# RUN
if __name__ == "__main__":
    total_t0 = time.time()  # NEW
    log_message("===== PIPELINE STARTED (5-band EEG) =====")  # CHANGED
    print("MULTIMODAL PREPROCESSING PIPELINE (5-band EEG)")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"EEG bands: {EEG_FREQ_BANDS}")  # NEW
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    print(f"fNIRS epoch: {FNIRS_EPOCH_TMAX}s")
    print(f"fNIRS sampling rate: {SFREQ_FNIRS} Hz")

    # NEW (verification run suggestion, per chat history): before running the full 26-subject
    # batch (which now costs 5x the spatial-mapping time per subject vs. broadband-only), run
    # on 2-3 subjects first and check the VERIFICATION CHECKLIST below.
    VERIFY_SUBJECTS_ONLY = False  # NEW: set True to run only VP001-VP003 for verification
    subject_range = range(1, 4) if VERIFY_SUBJECTS_ONLY else range(1, 13)  # NEW

    for sub_num in subject_range:  # CHANGED
        subject_id = f"VP{sub_num:03d}"
        process_subject(subject_id)

    print("ALL THE SELECTED SUBJECTS PROCESSED")

    # ==========================================================================================
    # NEW: VERIFICATION CHECKLIST (per chat history) -- run with VERIFY_SUBJECTS_ONLY=True first,
    # on 2-3 subjects, and check the following BEFORE running the full 26-subject batch:
    #   1. eeg_windows.shape == (N, 16, 16, 5, 600) for every processed subject (check the printed
    #      "EEG windows shape" line and each subject's _metadata.json -> eeg_shape).
    #   2. np.isnan(eeg_windows).any() == False and np.isinf(eeg_windows).any() == False.
    #   3. N (number of windows) for each verified subject matches the count from the PREVIOUS
    #      (broadband) preprocessing run for the same subject -- band decomposition/spatial
    #      mapping must not change which windows are extracted, only the EEG tensor's content.
    #   4. Per-band sanity: eeg_windows[..., i, :] should NOT be identical across i=0..4 (i.e. the
    #      5 bands are genuinely different signals, not 5 copies of broadband by mistake) --
    #      spot-check e.g. np.corrcoef of two bands on a few windows, correlation should be << 1.
    #   5. ICA component-removal count (printed "Removed N ICA components") compared against the
    #      old l_freq=1.0 runs for the same subjects -- flag if it changed drastically (see the
    #      l_freq 0.5 comment in preprocess_eeg for why this needs checking).
    #   6. Confirm eeg_spatial_mapping runtime per subject (~5x the old single-band time, as
    #      expected) is within an acceptable total for 26 subjects; if not, double-check that
    #      spatial_mapping_multiband() (not 5x spatial_mapping() calls) is actually being used.
    # ==========================================================================================


[2026-07-19 10:28:48] ===== PIPELINE STARTED (5-band EEG) =====
MULTIMODAL PREPROCESSING PIPELINE (5-band EEG)
EEG window: 3.0s, step: 1.0s
EEG bands: {'delta': (0.5, 4.0), 'theta': (4.0, 8.0), 'alpha': (8.0, 13.0), 'beta': (13.0, 30.0), 'gamma': (30.0, 40.0)}
fNIRS window: 3.0s, step: 1.0s
fNIRS epoch: 18.0s
fNIRS sampling rate: 10.0 Hz
[2026-07-19 10:28:48] START processing VP015
Processing VP015

 Loading EEG...
Loading EEG from: E:\kntu\Thesis\codes\DATA\VP015-EEG\cnt_nback.mat
EEG loaded: 388022 samples, 200.0 Hz, 567 markers
unique markers: [ 16  48  64  80  96 112 128 144]
EEG mnt_nback.mat loaded: pos_3d shape (3, 30)

 Loading and preprocessing fNIRS...
Loading raw fNIRS from: E:\kntu\Thesis\codes\DATA\VP015-NIRS\cnt_nback.mat
raw fNIRS sampling rate: 10.0 Hz
Raw fNIRS: oxy (20937, 36), deoxy (20937, 36)
Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...
Skipping wavelet/MARA/lowpass filtering (

In [ ]:
#2

In [ ]:
#version2 - session_id
# MULTIMODAL PREPROCESSING: EEG + fNIRS 
# Set the paths based on your system before running the code
# you can adjust which subjects to process at the bottom of this cell

import os
import numpy as np
import scipy.io as sio
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, resample
from scipy.interpolate import griddata
from scipy.interpolate import CloughTocher2DInterpolator  # NEW: used by the vectorized spatial_mapping()
from scipy.spatial import Delaunay  # NEW: used by the vectorized spatial_mapping()
import mne
import mne_icalabel
import pywt
import scipy.stats as stats
import pandas as pd
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt  # NEW: needed for before/after preprocessing plots
import time
import datetime
import json  # NEW: for per-subject metadata.json export (adopted from reviewing a third-party implementation, see chat history)
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"  # NEW: matches user's actual dataset folder
OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"  # NEW: matches user's actual output folder

# window parameters
EEG_WINDOW_LEN = 3.0      
EEG_STEP = 1.0            # 66% overlap
EEG_EPOCH_TMIN = 0.0
EEG_EPOCH_TMAX = 40.0      # NEW: cover the full 40 s task period of a single n-back trial

FNIRS_WINDOW_LEN = 3.0    # CHANGED: 10.0 -> 3.0 (length of EACH of the 11 segments, not the whole span)
FNIRS_NUM_SEGMENTS = 11   # NEW: number of overlapping 3s segments per EEG window, per the paper's approach
FNIRS_STEP = 1.0          # 90% overlap

BASELINE_TMIN = -5.0  # NEW: seconds before trial onset (start of baseline window)
BASELINE_TMAX = -2.0  # NEW: seconds before trial onset (end of baseline window)
FNIRS_EPOCH_TMIN = 0.0
FNIRS_EPOCH_TMAX = 18.0   # bacause of the hemodynamic delay

SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0      

EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}      # NEW: EEG session/trial-onset codes -> class
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}          # NEW: fNIRS session/trial-onset codes -> class

# ============================================================================
# NEW (5-band EEG decomposition, per MSTFDN Ding et al. 2026, Information Fusion):
# instead of feeding broadband EEG into spatial_mapping, the cleaned EEG signal
# (post-ICA, post-reref) is decomposed into 5 standard frequency bands BEFORE
# spatial mapping. Each band is spatially mapped independently and the 5 resulting
# 16x16 grids are stacked on a new trailing axis.
# ============================================================================
EEG_FREQ_BANDS = {
    'delta': (0.5, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta':  (13.0, 30.0),
    'gamma': (30.0, 40.0),
}
EEG_BAND_ORDER = ['delta', 'theta', 'alpha', 'beta', 'gamma']  # NEW: fixed order used everywhere bands are stacked


def apply_band_decomposition(raw_clean):
    """
    ورودی: raw_clean (سیگنال تمیزشده‌ی broadband، بعد از ICA، bad-channel interpolation و
    average re-reference -- یعنی خروجی preprocess_eeg، raw_eeg_clean در process_subject).
    خروجی: دیکشنری {نام_باند: MNE Raw object فیلترشده}.
    نکته: این تابع فقط تعریف می‌شود اینجا؛ فراخوانی واقعی‌اش در process_subject روی
    raw_eeg_clean انجام می‌شود (بعد از drop شدن VEOG/HEOG/TP9)، نه داخل preprocess_eeg،
    تا channel order با eeg_mnt (که بعد از preprocess_eeg ساخته می‌شود) هماهنگ بماند.
    """
    band_signals = {}
    for band_name, (l_freq, h_freq) in EEG_FREQ_BANDS.items():
        raw_band = raw_clean.copy().filter(
            l_freq=l_freq, h_freq=h_freq, method='fir', verbose=False
        )
        band_signals[band_name] = raw_band
    return band_signals


# LOAD EEG DATA
def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    
    print(f"Loading EEG from: {cnt_path}")
    
    # Load signal
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]
    cnt_key = real_cnt_keys[0]
    cnt_struct = mat_cnt[cnt_key]
    
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    
    if sfreq != SFREQ_EEG:
        print(f"Resampling from {sfreq} to {SFREQ_EEG} Hz")
        from scipy.signal import resample
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')
    
    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    real_mrk_keys = [k for k in mat_mrk.keys() if not k.startswith('__')]
    mrk_key = real_mrk_keys[0]
    mrk_struct = mat_mrk[mrk_key]
    
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    elif hasattr(mrk_struct, 'y'):
        y = mrk_struct.y
        if y.ndim > 1:
            if y.shape[0] == 1 or y.shape[1] == 1:
                event_codes = y.flatten()
            else:
                event_codes = np.argmax(y, axis=0) + 1
        else:
            event_codes = y.flatten()
    else:
        event_codes = event_positions.copy()
    
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes
    
    print(f"EEG loaded: {raw.n_times} samples, {sfreq} Hz, {len(events)} markers")
    print(f"unique markers: {np.unique(event_codes)}")

    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')  # NEW
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_eeg_mnt_keys = [k for k in mat_eeg_mnt.keys() if not k.startswith('__')]  # NEW
    eeg_mnt_struct = mat_eeg_mnt[real_eeg_mnt_keys[0]]  # NEW: has .pos_3d (3,30) and .clab (30,)
    print(f"EEG mnt_nback.mat loaded: pos_3d shape {np.asarray(eeg_mnt_struct.pos_3d).shape}")  # NEW

    return raw, events, sfreq, eeg_mnt_struct  # CHANGED: also return the real electrode-position struct


# LOAD AND PREPROCESS FNIRS DATA
def load_and_preprocess_fnirs(subject_id):
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")  # NEW: flat folder structure
    
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')  # NEW
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")
    
    print(f"Loading raw fNIRS from: {cnt_path}")
    
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]     # NEW
    cnt = mat_cnt[real_cnt_keys[0]]                                          # NEW

    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith('__')]     # NEW
    mnt = mat_mnt[real_mnt_keys[0]]                                          # NEW

    fnirs_events_time = None  # NEW
    fnirs_events_code = None  # NEW
    if os.path.exists(mrk_path):  # NEW
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)  # NEW
        real_fmrk_keys = [k for k in mat_fmrk.keys() if not k.startswith('__')]    # NEW
        fmrk = mat_fmrk[real_fmrk_keys[0]]                                         # NEW
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):  # NEW
            fnirs_events_time = np.asarray(fmrk.time).flatten()                     # NEW
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()               # NEW
    
    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0  # NEW
    print(f"raw fNIRS sampling rate: {sfreq_fnirs_raw} Hz")
    
    oxy_raw = np.asarray(cnt.oxy.x, dtype=np.float64)     # NEW
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)  # NEW
    
    print(f"Raw fNIRS: oxy {oxy_raw.shape}, deoxy {deoxy_raw.shape}")
    
    print("Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...")  # NEW
    hbo_raw, hbr_raw = oxy_raw, deoxy_raw  # NEW
    
    print("Skipping wavelet/MARA/lowpass filtering (MSTFDN-style: relying on the model's own large-kernel convolutions for implicit smoothing)...")  # NEW
    hbo_filtered, hbr_filtered = hbo_raw, hbr_raw  # NEW
    
    print(f"Resampling from {sfreq_fnirs_raw} Hz to {SFREQ_FNIRS} Hz...")
    from scipy.signal import resample
    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)
    
    print("Spatial mapping to 16x16...")
    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)
    
    print(f"fNIRS preprocessed: {fnirs_data.shape}")

    try:  # NEW
        qc_channel = 0  # NEW
        t_raw = np.arange(oxy_raw.shape[0]) / sfreq_fnirs_raw  # NEW
        t_proc = np.arange(hbo_resampled.shape[0]) / SFREQ_FNIRS  # NEW
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)  # NEW
        axes[0].plot(t_raw, oxy_raw[:, qc_channel], color='tab:red', linewidth=0.6)  # NEW
        axes[0].set_title(f"{subject_id} - fNIRS RAW HbO (channel {qc_channel}, before preprocessing)")  # NEW
        axes[0].set_xlabel("Time (s)")  # NEW
        axes[0].set_ylabel("HbO (raw units)")  # NEW
        axes[1].plot(t_proc, hbo_resampled[:, qc_channel], color='tab:blue', linewidth=0.8)  # NEW
        axes[1].set_title(f"{subject_id} - fNIRS PREPROCESSED HbO (channel {qc_channel}, after wavelet+MARA+lowpass+resample)")  # NEW
        axes[1].set_xlabel("Time (s)")  # NEW
        axes[1].set_ylabel("HbO (mmol/L)")  # NEW
        plt.tight_layout()  # NEW
        qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
        os.makedirs(qc_dir, exist_ok=True)  # NEW
        fig.savefig(os.path.join(qc_dir, f"{subject_id}_fnirs_before_after.png"), dpi=150)  # NEW
        plt.close(fig)  # NEW
        print(f"Saved fNIRS before/after QC plot for {subject_id}")  # NEW
    except Exception as e:  # NEW
        print(f"Could not save fNIRS QC plot for {subject_id}: {e}")  # NEW
    
    return fnirs_data, fnirs_events_time, fnirs_events_code  # CHANGED

# fNIRS HELPER FUNCTIONS
def calculate_hbo_hbr(oxy, deoxy):
    lambda1, lambda2 = 760, 850
    L = 3.0
    
    def get_dpf(lmbda, age=25):
        alpha, beta, gamma = 223.3, 0.05624, 0.8493
        delta, eps, zeta = -5.723e-7, 0.001245, -0.9025
        return alpha + beta * (age ** gamma) + delta * (lmbda ** 3) + eps * (lmbda ** 2) + zeta * lmbda
    
    dpf1 = get_dpf(lambda1)
    dpf2 = get_dpf(lambda2)
    
    e1_hbo, e1_hbr = 0.586, 1.54852
    e2_hbo, e2_hbr = 1.058, 0.69132
    
    denom = L * (e1_hbr * e2_hbo - e2_hbr * e1_hbo)
    hbo = (e1_hbr * (deoxy / dpf2) - e2_hbr * (oxy / dpf1)) / denom
    hbr = (e1_hbo * (oxy / dpf1) - e2_hbo * (deoxy / dpf2)) / denom
    
    return hbo, hbr

def wavelet_denoising(data, alpha=0.1, wavelet='db5'):
    cleaned = np.zeros_like(data)
    n_samples, n_channels = data.shape
    
    for ch in range(n_channels):
        signal = data[:, ch]
        level = min(int(np.floor(np.log2(n_samples))), 6)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        new_coeffs = [coeffs[0]]
        for i in range(1, len(coeffs)):
            d_coeffs = coeffs[i].copy()
            sigma = np.median(np.abs(d_coeffs - np.median(d_coeffs))) / 0.6745
            if sigma > 0:
                threshold = sigma * stats.norm.ppf(1 - alpha / 2)
                d_coeffs[np.abs(d_coeffs) > threshold] = 0
            new_coeffs.append(d_coeffs)
        cleaned[:, ch] = pywt.waverec(new_coeffs, wavelet)[:n_samples]
    
    return cleaned

def apply_mara(data, fs=10.0, window_size=41, threshold_factor=1.8):
    n_samples, n_channels = data.shape
    corrected = np.zeros_like(data)
    edge_len = int(0.5 * fs)
    
    for ch in range(n_channels):
        signal = data[:, ch].copy()
        msd = pd.Series(signal).rolling(window=window_size, center=True, min_periods=1).std().to_numpy()
        threshold = np.median(msd) + (threshold_factor * np.std(msd))
        ma_mask = (msd > threshold).astype(int)
        
        diff_mask = np.diff(ma_mask, prepend=0, append=0)
        starts = np.where(diff_mask == 1)[0]
        ends = np.where(diff_mask == -1)[0]
        indices = sorted(list(set([0, n_samples] + list(starts) + list(ends))))
        segments = [signal[indices[i]:indices[i+1]].copy() for i in range(len(indices) - 1)]
        
        if not segments:
            corrected[:, ch] = signal
            continue
        
        final_parts = [segments[0]]
        for i in range(1, len(segments)):
            prev_seg = final_parts[-1]
            curr_seg = segments[i]
            val_prev = np.median(prev_seg[-edge_len:]) if len(prev_seg) >= edge_len else prev_seg[-1]
            val_curr = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
            shift = val_prev - val_curr
            
            if ma_mask[indices[i]] == 1:
                x = np.arange(len(curr_seg))
                s_val = max(np.var(curr_seg) * len(curr_seg), 1e-6)
                try:
                    spline = UnivariateSpline(x, curr_seg, s=s_val)
                    curr_seg = curr_seg - spline(x)
                    val_curr_new = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
                    curr_seg += (val_prev - val_curr_new)
                except:
                    curr_seg += shift
            else:
                if abs(shift) > 2 * np.std(msd):
                    curr_seg += shift
            final_parts.append(curr_seg)
        
        full_signal = np.concatenate(final_parts)[:n_samples]
        t_idx = np.arange(n_samples)
        original_trend = np.polyval(np.polyfit(t_idx, data[:, ch], 1), t_idx)
        corrected_trend = np.polyval(np.polyfit(t_idx, full_signal, 1), t_idx)
        full_signal += (original_trend - corrected_trend)
        corrected[:, ch] = full_signal
    
    return corrected

def lowpass_filter(data, fs=10.0, cutoff=0.15):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(3, normal_cutoff, btype='low')
    filtered = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered[:, ch] = filtfilt(b, a, data[:, ch])
    return filtered

def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)  # NEW: shape (3, n_channels)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]  # NEW
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)  # NEW
    r_safe = np.where(r > 0, r, 1.0)  # NEW
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))  # NEW
    phi = np.arctan2(y3d, x3d)  # NEW
    x_coords = theta * np.cos(phi)  # NEW
    y_coords = theta * np.sin(phi)  # NEW
    
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, 
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    
    tri = Delaunay(points)  # NEW: triangulation depends only on `points`, computed ONCE instead of n_samples times
    values = data.T  # NEW: (n_samples, n_channels) -> (n_channels, n_samples)
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)  # NEW
    result = interp((grid_x, grid_y))  # NEW: shape (16, 16, n_samples)
    mapped = np.transpose(result, (2, 0, 1)).astype(np.float32)  # NEW: (16, 16, n_samples) -> (n_samples, 16, 16)
    
    return mapped


# NEW (5-band spatial mapping, optimized): instead of calling spatial_mapping() 5 separate
# times (one per band, each re-computing the Delaunay triangulation + CloughTocher setup from
# scratch), this batches all 5 bands' worth of time samples into ONE interpolator call, since
# CloughTocher2DInterpolator accepts `values` with arbitrary trailing dimensions as long as
# values.shape[0] == n_points. Triangulation and the (expensive) gradient-estimation/patch-setup
# step of CloughTocher are therefore computed ONCE for all 5 bands combined, instead of 5x.
# Verified to be mathematically identical to calling spatial_mapping() once per band and stacking
# the results (same triangulation, same interpolant, only how it's batched changes).
def spatial_mapping_multiband(band_data_list, mnt):
    """
    ورودی:
        band_data_list: لیستی از ۵ آرایه، هرکدوم به شکل (n_times, n_channels) -- یک آرایه به ازای
                         هر باند، به همون ترتیب EEG_BAND_ORDER.
        mnt: همون آبجکتی که spatial_mapping() هم استفاده می‌کنه (دارای .pos_3d).
    خروجی:
        آرایه‌ی (n_times, 16, 16, n_bands) -- دقیقاً معادل با اینکه spatial_mapping() رو
        جداگانه برای هر باند صدا بزنیم و با np.stack(..., axis=-1) ترکیب‌شون کنیم، ولی با
        triangulation و patch-setup مشترک (یک‌بار محاسبه به‌جای n_bands بار).
    """
    n_bands = len(band_data_list)
    n_samples, n_channels = band_data_list[0].shape

    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)
    r_safe = np.where(r > 0, r, 1.0)
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))
    phi = np.arctan2(y3d, x3d)
    x_coords = theta * np.cos(phi)
    y_coords = theta * np.sin(phi)

    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j,
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))

    # (n_channels, n_times, n_bands) -- values.shape[0] must equal n_points (n_channels)
    stacked = np.stack(band_data_list, axis=-1)          # (n_times, n_channels, n_bands)
    values = np.transpose(stacked, (1, 0, 2))              # (n_channels, n_times, n_bands)

    tri = Delaunay(points)                                  # computed ONCE for all bands
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)  # gradient/patch setup ONCE for all bands
    result = interp((grid_x, grid_y))                       # (16, 16, n_times, n_bands)
    mapped = np.transpose(result, (2, 0, 1, 3)).astype(np.float32)  # (n_times, 16, 16, n_bands)

    return mapped


class _EEGMontageAdapter:  # NEW
    """Minimal stand-in for the fNIRS .mat mnt struct, exposing only the .pos_3d attribute that
    spatial_mapping() reads."""
    def __init__(self, pos_3d):  # NEW
        self.pos_3d = pos_3d  # NEW: shape (3, n_channels), matching mnt.pos_3d's shape for fNIRS

def get_eeg_mnt(raw, eeg_mnt_struct):  # CHANGED
    """
    Builds an _EEGMontageAdapter (pos_3d shape (3, n_channels)) from the REAL electrode 3D
    positions in the dataset's own EEG mnt_nback.mat (eeg_mnt_struct.pos_3d / .clab), selecting
    and ordering them to match raw.ch_names exactly. Must be called AFTER any channel drops
    (VEOG/HEOG/TP9 in preprocess_eeg) so the returned position order matches the channel order of
    the data that will be passed to spatial_mapping()/spatial_mapping_multiband().
    """
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]  # NEW
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)  # NEW: shape (3, 30)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}  # NEW
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}  # NEW

    pos_list = []  # NEW
    missing = []  # NEW
    for ch in raw.ch_names:  # NEW
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))  # NEW
        if idx is None:  # NEW
            missing.append(ch)  # NEW
            pos_list.append([0.0, 0.0, 0.0])  # NEW
        else:  # NEW
            pos_list.append(full_pos_3d[:, idx])  # NEW

    if missing:  # NEW
        print(f"  WARNING: {len(missing)} EEG channel(s) not found in mnt_nback.mat's clab "  # NEW
              f"(spatial_mapping will place them at the grid center): {missing}")  # NEW

    pos_3d = np.array(pos_list, dtype=np.float64).T  # NEW: shape (3, n_channels)
    return _EEGMontageAdapter(pos_3d)  # NEW

# PREPROCESS EEG
def preprocess_eeg(raw, events):
    print("Preprocessing EEG...")

    raw_before_qc = raw.copy()  # NEW
    
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks='eeg', verbose=False)
    # FIX (5-band consistency, per chat history): l_freq lowered from 1.0 -> 0.5 Hz so the
    # broadband signal that band-decomposition later slices from actually CONTAINS the 0.5-1 Hz
    # content that the delta band (0.5-4 Hz, MSTFDN definition) needs. With l_freq=1.0, the delta
    # band filter would silently produce a 1-4 Hz band instead of the intended 0.5-4 Hz band,
    # since content below 1 Hz was already removed here. This also matches MSTFDN's own broadband
    # preprocessing range. Gamma (30-40 Hz) is unaffected since h_freq=40.0 is unchanged. ICA runs
    # AFTER this filter, so lowering l_freq may let more slow baseline-wander survive into ICA;
    # verify on 2-3 subjects that the number of ICA components removed doesn't change drastically
    # before running the full 26-subject batch (see verification checklist).
    raw_filtered.filter(l_freq=0.5, h_freq=40.0, method='fir', verbose=False)  # FIX: 1.0 -> 0.5
    
    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method='fir', verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ['VEOG', 'HEOG', 'TP9']:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    
    if bad_channels:
        print(f"Bad channels: {bad_channels}")
        raw_filtered.info['bads'] = bad_channels
    
    interpolated_channels = list(raw_filtered.info['bads'])  # NEW
    
    if raw_filtered.info['bads']:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    
    raw_filtered.set_eeg_reference('average', projection=False, verbose=False)
    
    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude='bads')

    print(f"ICA: using variance-based component selection (99% explained variance), "  # NEW
          f"picks={len(picks_for_ica)}, interpolated={len(interpolated_channels)}")     # NEW
    ica = mne.preprocessing.ICA(n_components=0.99, method='picard',                     # NEW
                                fit_params=dict(ortho=False, extended=True),            # NEW
                                random_state=42, verbose=False)                          # NEW
    ica.fit(raw_filtered, picks=picks_for_ica)

    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method='iclabel')

    ARTIFACT_CLASSES_IDX = {  # NEW
        'brain': 0, 'muscle artifact': 1, 'eye blink': 2,
        'heart beat': 3, 'line noise': 4, 'channel noise': 5, 'other': 6
    }
    ARTIFACT_TARGETS = ['eye blink', 'eye movement', 'muscle artifact', 'line noise']  # NEW
    PROB_THRESHOLD = 0.30  # NEW

    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels['labels'])
                           if label in ARTIFACT_TARGETS]  # CHANGED

    artifact_idx_prob = []  # NEW
    proba_matrix = ica.labels_scores_  # NEW
    for comp_idx in range(proba_matrix.shape[0]):  # NEW
        for cls_name in ARTIFACT_TARGETS:  # NEW
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)  # NEW
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:  # NEW
                artifact_idx_prob.append(comp_idx)  # NEW
                break  # NEW

    eog_indices_veog, eog_scores_veog = [], None  # NEW
    eog_indices_heog, eog_scores_heog = [], None  # NEW
    try:  # NEW
        if 'VEOG' in raw.ch_names:  # NEW
            eog_indices_veog, eog_scores_veog = ica.find_bads_eog(raw, ch_name='VEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(VEOG) flagged components: {eog_indices_veog}")  # NEW
        if 'HEOG' in raw.ch_names:  # NEW
            eog_indices_heog, eog_scores_heog = ica.find_bads_eog(raw, ch_name='HEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(HEOG) flagged components: {eog_indices_heog}")  # NEW
    except Exception as e:  # NEW
        print(f"  find_bads_eog failed: {e}")  # NEW

    BRAIN_PROB_CEILING = 0.5  # NEW
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX['brain']  # NEW
    
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))  # NEW
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}  # NEW
    if eog_only_rescued:  # NEW
        print(f"  Rescued from EOG-only rejection (ICLabel brain-prob >= {BRAIN_PROB_CEILING}): {sorted(eog_only_rescued)}")  # NEW
    
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )  # CHANGED

    all_class_names = ['brain', 'muscle artifact', 'eye blink', 'heart beat', 'line noise', 'channel noise', 'other']  # NEW
    for comp_idx in artifact_idx:  # NEW
        probs_str = ", ".join([f"{all_class_names[c]}={proba_matrix[comp_idx, c]:.2f}"  # NEW
                               for c in range(7)])  # NEW
        reason = []  # NEW
        if comp_idx in artifact_idx_argmax:  # NEW
            reason.append(f"argmax='{ic_labels['labels'][comp_idx]}'")  # NEW
        if comp_idx in artifact_idx_prob:  # NEW
            reason.append(f"prob>={PROB_THRESHOLD}")  # NEW
        if comp_idx in eog_indices_veog:  # NEW
            reason.append("VEOG-correlation")  # NEW
        if comp_idx in eog_indices_heog:  # NEW
            reason.append("HEOG-correlation")  # NEW
        print(f"  IC{comp_idx} flagged [{', '.join(reason)}]: {probs_str}")  # NEW

    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components")
    else:
        print("Removed 0 ICA components (no artifact ICs detected)")  # NEW

    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ['VEOG', 'HEOG', 'TP9']]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    try:  # NEW
        def _plot_all_channels_stacked(raw_obj, title, color, tmin=None, tmax=None):  # CHANGED
            sfreq_plot = raw_obj.info['sfreq']               # NEW
            if tmin is not None or tmax is not None:         # NEW
                start_sample = int((tmin or 0.0) * sfreq_plot)             # NEW
                stop_sample = int((tmax * sfreq_plot)) if tmax is not None else None  # NEW
                data = raw_obj.get_data(start=start_sample, stop=stop_sample)         # NEW
                t = (np.arange(data.shape[1]) + start_sample) / sfreq_plot            # NEW
            else:                                            # NEW
                data = raw_obj.get_data()                      # NEW
                t = np.arange(data.shape[1]) / sfreq_plot        # NEW
            ch_names_plot = raw_obj.ch_names                # NEW

            spread = np.median(np.abs(data - np.median(data, axis=1, keepdims=True))) * 30 + 1e-12  # NEW
            n_ch_plot = data.shape[0]                        # NEW
            fig_h = max(6, 0.5 * n_ch_plot)                 # NEW
            fig, ax = plt.subplots(figsize=(14, fig_h))      # NEW
            for i in range(n_ch_plot):                       # NEW
                offset = (n_ch_plot - 1 - i) * spread         # NEW
                ax.plot(t, data[i] + offset, color=color, linewidth=0.4)  # NEW
            ax.set_yticks([(n_ch_plot - 1 - i) * spread for i in range(n_ch_plot)])  # NEW
            ax.set_yticklabels(ch_names_plot, fontsize=6)    # NEW
            ax.set_xlabel("Time (s)")                        # NEW
            ax.set_ylabel("Channel")                         # NEW
            ax.set_title(title)                              # NEW
            plt.tight_layout()                                # NEW
            return fig                                        # NEW

        fig_before = _plot_all_channels_stacked(  # NEW
            raw_before_qc, "EEG RAW - All Channels (before preprocessing)", 'tab:red'  # NEW
        )  # NEW
        fig_after = _plot_all_channels_stacked(  # NEW
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA)", 'tab:blue'  # NEW
        )  # NEW
        fig_before_zoom = _plot_all_channels_stacked(  # NEW
            raw_before_qc, "EEG RAW - All Channels (before preprocessing, 10-30s)", 'tab:red', tmin=10.0, tmax=30.0  # NEW
        )  # NEW
        fig_after_zoom = _plot_all_channels_stacked(  # NEW
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA, 10-30s)", 'tab:blue', tmin=10.0, tmax=30.0  # NEW
        )  # NEW

        tmp_qc_dir = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc")  # NEW
        os.makedirs(tmp_qc_dir, exist_ok=True)  # NEW
        fig_before.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels.png"), dpi=150)  # NEW
        fig_after.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels.png"), dpi=150)  # NEW
        fig_before_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels_zoom10_30.png"), dpi=150)  # NEW
        fig_after_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels_zoom10_30.png"), dpi=150)  # NEW
        plt.close(fig_before)  # NEW
        plt.close(fig_after)  # NEW
        plt.close(fig_before_zoom)  # NEW
        plt.close(fig_after_zoom)  # NEW
    except Exception as e:  # NEW
        print(f"Could not save EEG QC plot: {e}")  # NEW
    
    return raw_filtered


def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,  # CHANGED
                                  fnirs_events_time, fnirs_events_code):  # CHANGED

    print("Extracting synchronized asymmetric sliding windows...")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    
    label_map = EEG_TRIAL_LABEL_MAP  # NEW
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())  # NEW
    target_events = events[np.isin(events[:, 2], valid_codes)]  # NEW
    
    print(f"found {len(target_events)} trial markers")
    if len(target_events) == 0:
        print(f"No trial markers found. Available markers: {np.unique(events[:, 2])}")
        return np.array([]), np.array([]), np.array([])

    fnirs_sync_ok = False  # NEW
    fnirs_trial_times = None  # NEW
    if fnirs_events_time is not None and fnirs_events_code is not None:  # NEW
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())  # NEW
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)  # NEW
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)  # NEW
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]  # NEW
        fnirs_order = np.argsort(fnirs_trial_times_ms)  # NEW
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0  # NEW
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])  # NEW

        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])  # NEW
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))  # NEW
        if (len(fnirs_trial_times) == len(target_events)  # NEW
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):  # NEW
            fnirs_sync_ok = True  # NEW
            print(f"fNIRS-own-marker sync check: OK ({len(fnirs_trial_times)} trials, class order matches EEG)")  # NEW
        else:
            print(f"  WARNING: fNIRS own markers do not match EEG 1:1 "  # NEW
                  f"(fNIRS n={len(fnirs_trial_times)}, EEG n={len(target_events)}) -- "  # NEW
                  f"falling back to EEG-clock-derived fNIRS timing for this subject.")  # NEW
    else:
        print("  WARNING: fNIRS event markers unavailable -- falling back to EEG-clock-derived fNIRS timing.")  # NEW
    
    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP
    
    eeg_windows = []
    fnirs_windows = []
    labels = []
    session_ids = []   # CROSS-SESSION
    trial_ids = []     # NEW: needed by training script's trial-level split (VPxxx_trial_ids.npy)
    
    for trial_idx, ev in enumerate(target_events):  # CHANGED
        event_sample_eeg = ev[0]
        raw_code = ev[2]
        
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        TRIALS_PER_SESSION = 9
        session_id = trial_idx // TRIALS_PER_SESSION + 1
        
        eeg_event_time = event_sample_eeg / sfreq_eeg  # CHANGED
        if fnirs_sync_ok:  # NEW
            fnirs_event_time = fnirs_trial_times[trial_idx]  # NEW
        else:  # NEW
            fnirs_event_time = eeg_event_time  # NEW

        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))  # NEW
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))    # NEW
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:  # NEW
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)  # NEW
        else:  # NEW
            fnirs_baseline = 0.0  # NEW

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            
            if w_end_eeg > eeg_raw.n_times:  # CHANGED
                continue

            fnirs_segments = []  # NEW
            valid_segments = True  # NEW
            for seg_idx in range(FNIRS_NUM_SEGMENTS):  # NEW
                seg_offset = offset_sec + seg_idx * FNIRS_STEP  # NEW
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))  # NEW
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs  # NEW

                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:  # NEW
                    valid_segments = False  # NEW
                    break  # NEW

                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)  # CHANGED

            if not valid_segments:  # NEW
                continue  # NEW
            
            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)  # CHANGED: QC-only now
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:  # CHANGED
                continue

            # CHANGED (5-band EEG): eeg_spatial_data is now (n_times, 16, 16, 5) -- one grid per
            # band, stacked on the trailing axis (see spatial_mapping_multiband() / EEG_BAND_ORDER).
            # Slicing along the time axis (axis 0) is unaffected by the extra trailing band axis.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # CHANGED: (600, 16, 16, 5)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 3, 0)          # CHANGED: (16, 16, 5, 600)

            fnirs_slice = np.stack(fnirs_segments, axis=0)      # NEW: (11, 30, 16, 16, 2)
            fnirs_slice = fnirs_slice.transpose(0, 2, 3, 1, 4)  # NEW: (11, 16, 16, 30, 2)
            
            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
            session_ids.append(session_id)   # CROSS-SESSION
            trial_ids.append(trial_idx)       # NEW
    
    eeg_windows = np.array(eeg_windows)
    fnirs_windows = np.array(fnirs_windows)
    labels = np.array(labels)
    session_ids = np.array(session_ids, dtype=np.int32)   # CROSS-SESSION
    trial_ids = np.array(trial_ids, dtype=np.int32)        # NEW
    
    print(f"Extracted {len(labels)} synchronized windows")
    if len(labels) > 0:
        print(f"Label distribution: {np.bincount(labels)}")
        print(f"EEG windows shape: {eeg_windows.shape} (expect (N,16,16,5,600))")  # NEW: sanity print
    
    return eeg_windows, fnirs_windows, labels, session_ids, trial_ids   # CHANGED: trial_ids added



LOG_FILE = os.path.join(OUTPUT_PATH, "processing_log.txt")  # NEW

def log_message(msg):  # NEW
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")



# MAIN PROCESSING FUNCTION
def process_subject(subject_id):
    t0 = time.time()  # NEW
    log_message(f"START processing {subject_id}")  # NEW
    print(f"Processing {subject_id}")
    
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")  # NEW
    
    if not os.path.exists(eeg_folder):
        log_message(f"SKIPPED {subject_id}: EEG folder not found")  # NEW
        print(f"EEG folder not found: {eeg_folder}")
        return
    
    try:
        print("\n Loading EEG...")
        raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)  # CHANGED
        
        print("\n Loading and preprocessing fNIRS...")
        fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(subject_id)  # CHANGED
        
        print("\n Normalizing fNIRS (subject-level, before windowing)...")  # NEW
        fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)  # NEW
        
        print("\n Preprocessing EEG...")
        raw_eeg_clean = preprocess_eeg(raw_eeg, events)

        # NEW (for eeg_preprocessing_qc.py, per chat history): save the continuous, channel-space,
        # post-notch/bandpass/CAR/ICA EEG signal as a .fif file, BEFORE band-decomposition/spatial-
        # mapping/windowing destroy the channel-space representation. This is the "exact" (non-lossy)
        # comparison source the QC script's load_clean_from_pipeline() looks for first
        # (OUTPUT_PATH/{subject_id}/{subject_id}_eeg_clean_raw.fif); without it, the QC script falls
        # back to a lossy approximation reconstructed from the windowed/spatial-mapped output.
        # sub_out_dir is normally created later (right before the .npy saves below); moved up here
        # (idempotent -- os.makedirs(..., exist_ok=True) further down is a harmless no-op re-call)
        # so this save can happen immediately, right after raw_eeg_clean is produced.
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW: moved up from below
        os.makedirs(sub_out_dir, exist_ok=True)  # NEW: moved up from below
        raw_eeg_clean.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_clean_raw.fif"),
                            overwrite=True)  # NEW
        print(f"Saved continuous cleaned EEG (for QC) to "
              f"{os.path.join(sub_out_dir, f'{subject_id}_eeg_clean_raw.fif')}")  # NEW

        eeg_full_data = raw_eeg_clean.get_data()  # NEW: kept for informational print only (channel-space scale)
        eeg_subject_mean = eeg_full_data.mean(axis=-1, keepdims=True)  # NEW
        eeg_subject_std = eeg_full_data.std(axis=-1, keepdims=True) + 1e-8  # NEW
        print(f"\n EEG per-subject (channel-space) stats: "  # NEW
              f"mean range [{eeg_subject_mean.min():.2e}, {eeg_subject_mean.max():.2e}], "  # NEW
              f"std range [{eeg_subject_std.min():.2e}, {eeg_subject_std.max():.2e}]")  # NEW

        # CHANGED (5-band EEG decomposition, per MSTFDN Ding et al. 2026): instead of spatially
        # mapping the single broadband signal, raw_eeg_clean is first decomposed into 5 standard
        # frequency bands (delta/theta/alpha/beta/gamma), each spatially mapped to a 16x16 grid,
        # and the 5 grids are stacked on a trailing axis -> (n_times, 16, 16, 5). Uses the batched
        # spatial_mapping_multiband() so the (expensive) Delaunay triangulation + CloughTocher
        # patch-setup is computed ONCE for all 5 bands instead of 5 separate times.
        print("\n Decomposing EEG into 5 frequency bands (delta/theta/alpha/beta/gamma)...")  # NEW
        band_signals = apply_band_decomposition(raw_eeg_clean)  # NEW: dict {band_name: Raw}

        eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)  # CHANGED: uses REAL mnt_nback.mat positions, matched by clab name

        print("\n Spatial mapping EEG (5 bands) to 16x16...")  # CHANGED
        band_data_list = []  # NEW
        for band_name in EEG_BAND_ORDER:  # NEW
            raw_band = band_signals[band_name]  # NEW
            band_data = raw_band.get_data()  # NEW: (n_channels, n_times)
            # per-channel z-score of this band's continuous signal, BEFORE spatial mapping,
            # mirroring the normalization already applied to broadband EEG in the previous pipeline
            band_data_normalized = (band_data - band_data.mean(axis=-1, keepdims=True)) / (  # NEW
                band_data.std(axis=-1, keepdims=True) + 1e-8)  # NEW
            band_data_list.append(band_data_normalized.T)  # NEW: (n_times, n_channels), matches spatial_mapping's expected input order

        eeg_spatial_full = spatial_mapping_multiband(band_data_list, eeg_mnt)  # NEW: (n_times, 16, 16, 5)
        print(f"EEG spatially mapped (5 bands): {eeg_spatial_full.shape}")  # NEW

        # per-band, per-grid-cell normalization of the spatially-mapped signal, applied to the
        # FULL continuous grid before windowing (mirrors fNIRS normalization above). axis=0 is
        # time, so mean/std are computed per (row, col, band) cell across the whole recording.
        eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (  # CHANGED
            eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)  # CHANGED

        try:  # NEW
            tmp_qc_before = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels.png")  # NEW
            tmp_qc_after = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels.png")    # NEW
            tmp_qc_before_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels_zoom10_30.png")  # NEW
            tmp_qc_after_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels_zoom10_30.png")    # NEW
            if os.path.exists(tmp_qc_before) and os.path.exists(tmp_qc_after):  # NEW
                final_qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
                os.makedirs(final_qc_dir, exist_ok=True)  # NEW
                final_qc_before = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels.png")  # NEW
                final_qc_after = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels.png")    # NEW
                os.replace(tmp_qc_before, final_qc_before)  # NEW
                os.replace(tmp_qc_after, final_qc_after)    # NEW
                if os.path.exists(tmp_qc_before_zoom) and os.path.exists(tmp_qc_after_zoom):  # NEW
                    final_qc_before_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels_zoom10_30.png")  # NEW
                    final_qc_after_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels_zoom10_30.png")    # NEW
                    os.replace(tmp_qc_before_zoom, final_qc_before_zoom)  # NEW
                    os.replace(tmp_qc_after_zoom, final_qc_after_zoom)    # NEW
                print(f"Saved EEG before/after (all-channels) QC plots for {subject_id}")  # NEW
        except Exception as e:  # NEW
            print(f"Could not finalize EEG QC plot for {subject_id}: {e}")  # NEW
        
        print("\n Extracting synchronized asymmetric windows...")
        eeg_windows, fnirs_windows, labels, session_ids, trial_ids = extract_synchronized_windows(   # CHANGED: trial_ids added
            raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg,
            fnirs_events_time, fnirs_events_code
        )
        
        if len(labels) == 0:
            log_message(f"WARNING {subject_id}: no windows extracted")  # NEW
            print(f"No windows extracted for {subject_id}")
            return
        
        print("\n Normalizing...")
        # NOTE: EEG normalization already happened above (per-band, per-grid-cell, before
        # windowing) -- eeg_windows already carries normalized values, no further step needed here.
        
        print("\n Saving...")
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
        os.makedirs(sub_out_dir, exist_ok=True)
        
        np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)    # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)  # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)               # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_session_ids.npy"), session_ids)   # CROSS-SESSION
        np.save(os.path.join(sub_out_dir, f"{subject_id}_trial_ids.npy"), trial_ids)         # NEW

        metadata = {  # NEW
            'subject_id': subject_id,  # NEW
            'eeg_shape': list(eeg_windows.shape),  # NEW: expect (N,16,16,5,600)
            'fnirs_shape': list(fnirs_windows.shape),  # NEW
            'labels_distribution': np.bincount(labels).tolist(),  # NEW
            'eeg_window_len': EEG_WINDOW_LEN,  # NEW
            'eeg_step': EEG_STEP,  # NEW
            'eeg_epoch_tmax': EEG_EPOCH_TMAX,  # NEW
            'eeg_freq_bands': EEG_FREQ_BANDS,  # NEW
            'eeg_band_order': EEG_BAND_ORDER,  # NEW
            'eeg_broadband_lfreq': 0.5,  # NEW: documents the 1.0 -> 0.5 fix
            'fnirs_num_segments': FNIRS_NUM_SEGMENTS,  # NEW
            'fnirs_window_len': FNIRS_WINDOW_LEN,  # NEW
            'fnirs_step': FNIRS_STEP,  # NEW
            'sfreq_eeg': SFREQ_EEG,  # NEW
            'sfreq_fnirs': SFREQ_FNIRS,  # NEW
            'beer_lambert_applied': False,  # NEW
            'fnirs_sync_fix_applied': True,  # NEW
            'normalization': 'per-subject z-score (EEG grid per-band + fNIRS grid, before windowing)',  # CHANGED
        }
        with open(os.path.join(sub_out_dir, f"{subject_id}_metadata.json"), 'w') as f:  # NEW
            json.dump(metadata, f, indent=2)
        
        print(f"\n {subject_id} completed!")
        print(f"EEG windows: {eeg_windows.shape}")
        print(f"fNIRS windows: {fnirs_windows.shape}")
        print(f"Labels: {np.bincount(labels)}")
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"DONE {subject_id} in {elapsed:.1f} min | EEG: {eeg_windows.shape} | fNIRS: {fnirs_windows.shape} | Labels: {np.bincount(labels)}")  # NEW
        
    except Exception as e:
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"ERROR processing {subject_id} after {elapsed:.1f} min: {str(e)}")  # NEW
        print(f"Error processing {subject_id}: {str(e)}")
        import traceback
        traceback.print_exc()

# RUN
if __name__ == "__main__":
    total_t0 = time.time()  # NEW
    log_message("===== PIPELINE STARTED (5-band EEG) =====")  # CHANGED
    print("MULTIMODAL PREPROCESSING PIPELINE (5-band EEG)")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"EEG bands: {EEG_FREQ_BANDS}")  # NEW
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    print(f"fNIRS epoch: {FNIRS_EPOCH_TMAX}s")
    print(f"fNIRS sampling rate: {SFREQ_FNIRS} Hz")

    # NEW (verification run suggestion, per chat history): before running the full 26-subject
    # batch (which now costs 5x the spatial-mapping time per subject vs. broadband-only), run
    # on 2-3 subjects first and check the VERIFICATION CHECKLIST below.
    VERIFY_SUBJECTS_ONLY = False  # NEW: set True to run only VP001-VP003 for verification
    subject_range = range(1, 4) if VERIFY_SUBJECTS_ONLY else range(1, 27)  # NEW

    for sub_num in subject_range:  # CHANGED
        subject_id = f"VP{sub_num:03d}"
        process_subject(subject_id)

    print("ALL THE SELECTED SUBJECTS PROCESSED")

    # ==========================================================================================
    # NEW: VERIFICATION CHECKLIST (per chat history) -- run with VERIFY_SUBJECTS_ONLY=True first,
    # on 2-3 subjects, and check the following BEFORE running the full 26-subject batch:
    #   1. eeg_windows.shape == (N, 16, 16, 5, 600) for every processed subject (check the printed
    #      "EEG windows shape" line and each subject's _metadata.json -> eeg_shape).
    #   2. np.isnan(eeg_windows).any() == False and np.isinf(eeg_windows).any() == False.
    #   3. N (number of windows) for each verified subject matches the count from the PREVIOUS
    #      (broadband) preprocessing run for the same subject -- band decomposition/spatial
    #      mapping must not change which windows are extracted, only the EEG tensor's content.
    #   4. Per-band sanity: eeg_windows[..., i, :] should NOT be identical across i=0..4 (i.e. the
    #      5 bands are genuinely different signals, not 5 copies of broadband by mistake) --
    #      spot-check e.g. np.corrcoef of two bands on a few windows, correlation should be << 1.
    #   5. ICA component-removal count (printed "Removed N ICA components") compared against the
    #      old l_freq=1.0 runs for the same subjects -- flag if it changed drastically (see the
    #      l_freq 0.5 comment in preprocess_eeg for why this needs checking).
    #   6. Confirm eeg_spatial_mapping runtime per subject (~5x the old single-band time, as
    #      expected) is within an acceptable total for 26 subjects; if not, double-check that
    #      spatial_mapping_multiband() (not 5x spatial_mapping() calls) is actually being used.
    # ==========================================================================================

[2026-07-19 12:10:38] ===== PIPELINE STARTED (5-band EEG) =====
MULTIMODAL PREPROCESSING PIPELINE (5-band EEG)
EEG window: 3.0s, step: 1.0s
EEG bands: {'delta': (0.5, 4.0), 'theta': (4.0, 8.0), 'alpha': (8.0, 13.0), 'beta': (13.0, 30.0), 'gamma': (30.0, 40.0)}
fNIRS window: 3.0s, step: 1.0s
fNIRS epoch: 18.0s
fNIRS sampling rate: 10.0 Hz
[2026-07-19 12:10:38] START processing VP001
Processing VP001

 Loading EEG...
Loading EEG from: E:\kntu\Thesis\codes\DATA\VP001-EEG\cnt_nback.mat
EEG loaded: 394930 samples, 200.0 Hz, 567 markers
unique markers: [ 16  48  64  80  96 112 128 144]
EEG mnt_nback.mat loaded: pos_3d shape (3, 30)

 Loading and preprocessing fNIRS...
Loading raw fNIRS from: E:\kntu\Thesis\codes\DATA\VP001-NIRS\cnt_nback.mat
raw fNIRS sampling rate: 10.0 Hz
Raw fNIRS: oxy (21178, 36), deoxy (21178, 36)
Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...
Skipping wavelet/MARA/lowpass filtering (